# 04 — Results Analysis

Load experiment results and generate comparison plots and tables.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, '..')
from koopman_eml.ctf import radar_plot

## 1. Load Results

In [ ]:
results_dir = Path('../results/ctf_lorenz')

methods = {}
if results_dir.exists():
    for subdir in sorted(results_dir.iterdir()):
        if subdir.is_dir():
            mf = subdir / 'metrics.json'
            if mf.exists():
                with open(mf) as f:
                    methods[subdir.name] = json.load(f)

if methods:
    print(f'Loaded results for {len(methods)} methods:')
    for name, m in methods.items():
        print(f'  {name}: E1={m.get("E1", "N/A")}')
else:
    print('No results found. Run experiments first:\n'
          '  python experiments/ctf_lorenz/run_eml.py\n'
          '  python experiments/ctf_lorenz/run_edmd.py\n'
          '  python experiments/ctf_lorenz/run_deep.py')

## 2. CTF Leaderboard Comparison

In [ ]:
CTF_LEADERBOARD = {
    'LSTM': 64.54,
    'DeepONet': 57.80,
    'Reservoir': 54.87,
    'KAN': 47.28,
    'SINDy': -3.00,
    'PyKoopman': -20.11,
}

all_names = []
all_e1 = []
all_colors = []

for name, m in methods.items():
    all_names.append(name)
    all_e1.append(m.get('E1', 0))
    all_colors.append('#2d5a27')

for name, score in CTF_LEADERBOARD.items():
    all_names.append(f'{name} (pub.)')
    all_e1.append(score)
    all_colors.append('#8b3a3a')

if all_names:
    order = np.argsort(all_e1)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh([all_names[i] for i in order],
            [all_e1[i] for i in order],
            color=[all_colors[i] for i in order])
    ax.set_xlabel('E1 Score')
    ax.set_title('CTF Lorenz: Short-Term Forecasting (E1)')
    ax.axvline(0, color='k', lw=0.5)
    plt.tight_layout()
    plt.show()

## 3. Prediction Rollout Comparison

In [ ]:
# Load prediction arrays if available
pred_arrays = {}
if results_dir.exists():
    for subdir in results_dir.iterdir():
        pf = subdir / 'predictions.npy'
        if pf.exists():
            pred_arrays[subdir.name] = np.load(pf)

if pred_arrays:
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    n_show = 300
    labels = ['x', 'y', 'z']

    for name, pred in pred_arrays.items():
        for i, ax in enumerate(axes):
            if pred.shape[1] > i:
                ax.plot(pred[:n_show, i], label=name, alpha=0.8)

    for i, ax in enumerate(axes):
        ax.set_ylabel(labels[i])
        ax.legend(loc='upper right', fontsize=8)
    axes[0].set_title('Prediction Rollout Comparison')
    axes[-1].set_xlabel('Time step')
    plt.tight_layout()
    plt.show()
else:
    print('No prediction arrays found.')

## 4. Summary Table

In [ ]:
if methods:
    print(f'{"Method":<25} {"RMSE":>10} {"Valid Steps":>12} {"E1":>8}')
    print('-' * 60)
    for name, m in sorted(methods.items(), key=lambda x: -x[1].get('E1', 0)):
        print(f'{name:<25} {m.get("rmse", 0):>10.4f} {m.get("valid_prediction_steps", 0):>12} {m.get("E1", 0):>8.2f}')
    print('-' * 60)
    print('\nCTF Leaderboard (published averages):')
    for name, score in sorted(CTF_LEADERBOARD.items(), key=lambda x: -x[1]):
        print(f'  {name:<20} avg: {score:.2f}')
else:
    print('No results to display.')